# Sampling physical gapmoe parameters with emcee

This notebook samples the canonical physical parameters with the NumPy histogram backend:

- `ML`: lens mass in solar masses
- `DL`: lens distance in kpc
- `DS`: source distance in kpc
- `mu_N`: heliocentric relative proper motion north in mas/yr
- `mu_E`: heliocentric relative proper motion east in mas/yr

**Prerequisites**: run `pre_runner.ipynb` first to generate the histogram files
in `example/pre_runner_outputs/demo/`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "gapmoe").exists():
    repo_root = cwd
elif (cwd.parent / "src" / "gapmoe").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or example directory.")

sys.path.insert(0, str(repo_root / "src"))

import emcee
import matplotlib.pyplot as plt
import numpy as np

from gapmoe import GalacticPrior, HistogramDensity

rng = np.random.default_rng(12345)
repo_root

## Load pre-generated histogram files

In [ ]:
demo_dir = repo_root / "example" / "pre_runner_outputs" / "demo"
if not demo_dir.exists():
    raise FileNotFoundError(f"{demo_dir} not found. Run pre_runner.ipynb first.")

density = HistogramDensity.from_paths(
    demo_dir / "mass.dat",
    demo_dir / "rho.dat",
    demo_dir / "murel.dat",
)
prior = GalacticPrior(density)

trial = (0.3, 5.0, 8.0, 5.0, 2.0)
prior.log_prob(*trial)

## Define the emcee target

This samples the Galactic prior itself. For a real event, add the light-curve likelihood to `log_probability`.

In [ ]:
bounds = {
    "ML": (0.08, 1.5),
    "DL": (0.1, 12.0),
    "DS": (0.5, 16.0),
    "mu_N": (-30.0, 30.0),
    "mu_E": (-30.0, 30.0),
}

def in_bounds(theta):
    ML, DL, DS, mu_N, mu_E = theta
    return (
        bounds["ML"][0] <= ML <= bounds["ML"][1]
        and bounds["DL"][0] <= DL <= bounds["DL"][1]
        and bounds["DS"][0] <= DS <= bounds["DS"][1]
        and bounds["mu_N"][0] <= mu_N <= bounds["mu_N"][1]
        and bounds["mu_E"][0] <= mu_E <= bounds["mu_E"][1]
        and DS > DL
    )

def log_probability(theta):
    if not in_bounds(theta):
        return -np.inf
    return prior.log_prob(*theta)

In [ ]:
def draw_finite_initial_positions(nwalkers, max_tries=100000):
    positions = []
    while len(positions) < nwalkers and max_tries > 0:
        max_tries -= 1
        theta = np.array([
            rng.uniform(*bounds["ML"]),
            rng.uniform(*bounds["DL"]),
            rng.uniform(*bounds["DS"]),
            rng.uniform(*bounds["mu_N"]),
            rng.uniform(*bounds["mu_E"]),
        ])
        lp = log_probability(theta)
        if np.isfinite(lp):
            positions.append(theta)
    if len(positions) != nwalkers:
        raise RuntimeError("Could not find enough finite initial walker positions.")
    return np.array(positions)

ndim = 5
nwalkers = 32
initial = draw_finite_initial_positions(nwalkers)
initial[:3]

In [ ]:
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(initial, 1000, progress=True)

acceptance_fraction = np.mean(sampler.acceptance_fraction)
acceptance_fraction

In [ ]:
chain = sampler.get_chain(discard=200, thin=5, flat=True)
labels = ["ML", "DL", "DS", "mu_N", "mu_E"]

for i, label in enumerate(labels):
    q16, q50, q84 = np.percentile(chain[:, i], [16, 50, 84])
    print(f"{label:>4s}: {q50:.4g} -{q50 - q16:.4g} +{q84 - q50:.4g}")

In [ ]:
fig, axes = plt.subplots(ndim, 1, figsize=(9, 7), sharex=True)
raw_chain = sampler.get_chain()

for i, ax in enumerate(axes):
    ax.plot(raw_chain[:, :, i], color="black", alpha=0.2, lw=0.5)
    ax.set_ylabel(labels[i])

axes[-1].set_xlabel("step")
fig.tight_layout()

In [ ]:
import corner
from matplotlib.lines import Line2D

genulens_path = repo_root / "example" / "genulens_out.dat"
genulens = np.genfromtxt(genulens_path, comments="#")
genulens = np.atleast_2d(genulens)

genulens_weights = genulens[:, 0]
genulens_piE = genulens[:, 6]
genulens_mu_rel = genulens[:, 9]
genulens_chain = np.column_stack([
    genulens[:, 1],          # ML [Msun]
    genulens[:, 2] / 1000.0, # DL [kpc]
    genulens[:, 3] / 1000.0, # DS [kpc]
    genulens_mu_rel * genulens[:, 7] / genulens_piE,
    genulens_mu_rel * genulens[:, 8] / genulens_piE,
])

valid = (
    np.all(np.isfinite(genulens_chain), axis=1)
    & np.isfinite(genulens_weights)
    & (genulens_weights > 0.0)
    & (genulens_piE > 0.0)
    & (genulens_mu_rel < 20.0)
    & (genulens[:, 1] < 1.2)
)
genulens_chain = genulens_chain[valid]
genulens_weights = genulens_weights[valid]

fig = corner.corner(
    chain,
    labels=labels,
    hist_kwargs={"density": True},
)

corner.corner(
    genulens_chain,
    labels=labels,
    weights=genulens_weights,
    fig=fig,
    color="tab:orange",
    plot_datapoints=False,
    fill_contours=False,
    hist_kwargs={"density": True},
)

legend_handles = [
    Line2D([0], [0], color="C0", lw=2, label="gapmoe"),
    Line2D([0], [0], color="tab:orange", lw=2, label="genulens"),
]
fig.legend(handles=legend_handles, loc="upper right")
plt.show()
